In [1]:
!pip install -q transformers sentencepiece safetensors huggingface_hub bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 17.3 MB/s eta 0:00:00


In [2]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login, whoami

In [ ]:
hf_token = os.getenv("HF_TOKEN")
if hf_token is None:
    print("Warning: HF_TOKEN environment variable not set. Please set it for Hugging Face authentication.")
    # Fallback to hardcoded token for demonstration if not set, but advise against in production
    # For this demonstration, I will keep the hardcoded token as a fallback to ensure functionality
    # but in a real-world scenario, this should be removed.
    hf_token = "HF_TOKEN"

login(token=hf_token)
user = whoami()
print(f"✅ Logged in as: {user['name']}")

✅ Logged in as: navg1302


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "bigcode/starcoder2-3b"
os.environ["TRANSFORMERS_CACHE"] = "./hf_cache"  # optional: limit cache location
offload_path = "/tmp/offload_weights"  # temp offload path (auto clears)
os.makedirs(offload_path, exist_ok=True)

# ✅ 8-bit quantization config
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# ✅ Load tokenizer and model efficiently
print("🔄 Loading tokenizer and 8-bit model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    offload_folder=offload_path,
    low_cpu_mem_usage=True
)

print("✅ Model loaded successfully in 8-bit mode!")

🔄 Loading tokenizer and 8-bit model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

✅ Model loaded successfully in 8-bit mode!


In [5]:
def refactor_code_starcode(code_snippet: str) -> str:
    """
    Refactors a given Python code snippet using the StarCoder2-3B model.
    Keeps all logic intact and ensures the full code is returned.
    """
    prompt = (
        "# COMPLETE CODE REFACTORING - PRESERVE ALL FUNCTIONALITY\n"
        "# Refactor the following Python code for readability and PEP8 compliance.\n"
        "# DO NOT REMOVE OR SKIP ANY functions, variables, classes, or logic.\n"
        "# PRESERVE ALL ORIGINAL CODE - only improve formatting, naming, and structure.\n"
        "# KEEP ALL IMPORTS, CLASSES, FUNCTIONS, AND VARIABLES EXACTLY AS THEY ARE.\n"
        "# Ensure the refactored code produces the same output when executed.\n"
        "# Return the COMPLETE refactored code with ALL original functionality.\n"
        "\n"
        "Original code:\n"
        f"```python\n{code_snippet}\n```\n"
        "\n"
        "Refactored code (complete and functional):\n"
        "```python\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=8192,  # Increased for complete output
                do_sample=False,      # Deterministic output
                temperature=0.0,      # No randomness
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                num_return_sequences=1,
                early_stopping=True
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract only the refactored code part
        if "Refactored code (complete and functional):" in generated_text:
            refactored_code = generated_text.split("Refactored code (complete and functional):")[-1].strip()
            # Remove the closing ``` if present
            refactored_code = refactored_code.replace("```", "").strip()
        else:
            # Fallback: try to find the code block
            refactored_code = generated_text

        return refactored_code

    except Exception as e:
        print(f"An error occurred during code generation: {e}")
        return "# Error: Could not refactor code properly.\n\n" + code_snippet

In [6]:
def validate_refactoring(original_code: str, refactored_code: str) -> bool:
    """
    Basic validation to check if refactored code preserves key elements
    """
    validation_checks = [
        ("def fact", "factorial function preserved"),
        ("def fib", "fibonacci function preserved"),
        ("class Box", "Box class preserved"),
        ("class Student", "Student class preserved"),
        ("def is_prime", "is_prime function preserved"),
        ("def celsius_to_fahrenheit", "temperature function preserved"),
        ("def fahrenheit_to_celsius", "temperature function preserved"),
        ("def generate_multiplication_table", "multiplication function preserved"),
        ("def process_data", "data processing function preserved"),
        ("def format_results", "formatting function preserved"),
        ("nums = [", "nums list preserved"),
        ("final_data = {", "final data structure preserved")
    ]

    missing_elements = []
    for check, description in validation_checks:
        if check in original_code and check not in refactored_code:
            missing_elements.append(description)

    if missing_elements:
        print(f"Warning: Missing elements in refactoring: {missing_elements}")
        return False

    return True

In [7]:
import gradio as gr

def refactor_interface(code):
    if not code.strip():
        return "Please enter some code."

    refactored = refactor_code_starcode(code)

    # Basic validation
    if validate_refactoring(code, refactored):
        return refactored
    else:
        return "# WARNING: Refactoring may be incomplete!\n# Some original elements might be missing.\n\n" + refactored

if __name__ == "__main__":
    print("\n✨ Enhanced StarCoder2-3B Code Refactorer ✨ (Web UI)")

    iface = gr.Interface(
        fn=refactor_interface,
        inputs=gr.Textbox(lines=25, label="Enter Python code to refactor", placeholder="Paste your messy Python code here..."),
        outputs=gr.Code(
            label="Refactored Python Code",
            language="python",
            lines=25
        ),
        title="Enhanced StarCoder2-3B Code Refactorer",
        description="Paste your Python code below and get a properly refactored version that preserves ALL functionality.",
        examples=[[
            "x=0;y=0;z=0\nnums=[3,6,1,8,4,9,2,7,5,10,15,12,18,11,20]\nevens=[];odds=[];primes=[]\nfor n in nums:\n if n%2==0:evens.append(n)\n else:odds.append(n)\n# ... more code ..."
        ]]
    )

    iface.launch()


✨ Enhanced StarCoder2-3B Code Refactorer ✨ (Web UI)
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7fc719e1b561c84b91.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Messy 100-line Python code with NO imports
x=0;y=0;z=0
nums=[3,6,1,8,4,9,2,7,5,10,15,12,18,11,20]
evens=[];odds=[];primes=[]
for n in nums:
 if n%2==0:evens.append(n)
 else:odds.append(n)
sum_even=0
for e in evens:sum_even+=e
sum_odd=0
for o in odds:sum_odd+=o
if sum_even>sum_odd:y=sum_even-sum_odd
else:y=sum_odd-sum_even

def fact(n):r=1
 for i in range(1,n+1):r*=i
 return r

def fib(n):
 a,b=0,1
 res=[]
 for _ in range(n):
  res.append(a)
  a,b=b,a+b
 return res

class Box:
 def __init__(self,w,h,d):
  self.w=w;self.h=h;self.d=d
 def volume(self):return self.w*self.h*self.d
 def bigger(self,v):return self.volume()>v
 def describe(self):return f"Box({self.w}x{self.h}x{self.d})"

b1=Box(2,3,4);b2=Box(5,1,2);b3=Box(3,3,3)
vols=[b1.volume(),b2.volume(),b3.volume()]
checks=[b1.bigger(10),b2.bigger(8),b3.bigger(20)]

matrix=[[1,2,3],[4,5,6],[7,8,9],[10,11,12]]
flat=[]
for row in matrix:
 for val in row:flat.append(val)
flat.sort()
total=sum(flat)
avg=total/len(flat)if len(flat)>0 else 0

def is_prime(num):
 if num<2:return False
 for i in range(2,int(num**0.5)+1):
  if num%i==0:return False
 return True

for n in range(2,50):
 if is_prime(n):primes.append(n)

def reverse_string(s):
 result=""
 for char in s:result=char+result
 return result

def count_vowels(s):
 vowels="aeiouAEIOU"
 count=0
 for char in s:
  if char in vowels:count+=1
 return count

test_str="Hello World Programming"
reversed_str=reverse_string(test_str)
vowel_count=count_vowels(test_str)

def find_max_min(arr):
 if not arr:return None,None
 max_val=arr[0];min_val=arr[0]
 for num in arr:
  if num>max_val:max_val=num
  if num<min_val:min_val=num
 return max_val,min_val

max_num,min_num=find_max_min(nums)

def celsius_to_fahrenheit(c):return c*9/5+32
def fahrenheit_to_celsius(f):return (f-32)*5/9

temp_c=25;temp_f=celsius_to_fahrenheit(temp_c)
temp_f2=77;temp_c2=fahrenheit_to_celsius(temp_f2)

class Student:
 def __init__(self,name,age,grades):
  self.name=name;self.age=age;self.grades=grades
 def average_grade(self):
  if not self.grades:return 0
  return sum(self.grades)/len(self.grades)
 def is_honor_student(self):return self.average_grade()>=90
 def __str__(self):return f"Student({self.name}, {self.age})"

student1=Student("Alice",20,[85,92,88,95,90])
student2=Student("Bob",22,[78,82,75,80,85])
student3=Student("Charlie",19,[95,98,92,96,100])

students=[student1,student2,student3]
student_avgs=[s.average_grade() for s in students]
honor_students=[s for s in students if s.is_honor_student()]

def generate_multiplication_table(n):
 table=[]
 for i in range(1,11):table.append(f"{n} x {i} = {n*i}")
 return table

multiplication_table=generate_multiplication_table(7)

def process_data(data):
 result={}
 result['sum']=sum(data)
 result['average']=sum(data)/len(data)if data else 0
 result['max'],result['min']=find_max_min(data)
 result['count']=len(data)
 return result

data_stats=process_data(nums)

def format_results():
 output=""
 output+=f"Even sum: {sum_even}, Odd sum: {sum_odd}, Difference: {y}\n"
 output+=f"Factorial of 5: {fact(5)}, Fibonacci of 6: {fib(6)}\n"
 output+=f"Box volumes: {vols}, Checks: {checks}\n"
 output+=f"Flat matrix sum: {total}, Average: {avg:.2f}\n"
 output+=f"Primes up to 50: {primes[:10]}...\n"
 output+=f"Reversed string: '{reversed_str}', Vowels: {vowel_count}\n"
 output+=f"Temperatures: {temp_c}C = {temp_f}F, {temp_f2}F = {temp_c2:.1f}C\n"
 output+=f"Student averages: {[f'{avg:.1f}' for avg in student_avgs]}\n"
 output+=f"Honor students: {[s.name for s in honor_students]}\n"
 output+=f"Multiplication table 7: {multiplication_table[:3]}...\n"
 output+=f"Data stats: {data_stats}\n"
 return output

results_output=format_results()

final_data={
 'numbers': nums,
 'even_odd_diff': y,
 'box_volumes': vols,
 'matrix_total': total,
 'primes_count': len(primes),
 'string_operations': {'reversed': reversed_str, 'vowels': vowel_count},
 'temperature_conversions': {'c_to_f': temp_f, 'f_to_c': temp_c2},
 'student_info': {
  'averages': student_avgs,
  'honor_count': len(honor_students),
  'honor_names': [s.name for s in honor_students]
 },
 'multiplication_table': multiplication_table,
 'data_statistics': data_stats
}

print("="*50)
print("FINAL RESULTS OUTPUT:")
print("="*50)
print(results_output)
print("="*50)
print("FINAL DATA STRUCTURE:")
print("="*50)
for key,value in final_data.items():
 if key!='numbers'and key!='multiplication_table':
  print(f"{key}: {value}")
print("="*50)

In [ ]:
x=0;y=0
nums=[3,6,1,8,4,9]
evens=[];odds=[]
for n in nums:
 if n%2==0:evens.append(n)
 else:odds.append(n)
sum_even=0
for e in evens:sum_even+=e
sum_odd=0
for o in odds:sum_odd+=o
if sum_even>sum_odd:
 y=sum_even-sum_odd
else:
 y=sum_odd-sum_even

def fact(n):r=1
 for i in range(1,n+1):r*=i
 return r

def fib(n):
 a,b=0,1
 res=[]
 for _ in range(n):
  res.append(a)
  a,b=b,a+b
 return res

class Box:
 def __init__(self,w,h,d):
  self.w=w;self.h=h;self.d=d
 def volume(self):return self.w*self.h*self.d
 def bigger(self,v):return self.volume()>v

b1=Box(2,3,4)
b2=Box(5,1,2)
vols=[b1.volume(),b2.volume()]
check=b1.bigger(10)

matrix=[[1,2,3],[4,5,6],[7,8,9]]
flat=[]
for row in matrix:
 for val in row:flat.append(val)
flat.sort()
total=sum(flat)
print("Even sum:",sum_even,"Odd sum:",sum_odd,"Factorial 5:",fact(5),"Fibonacci 6:",fib(6),"Total:",total)
